In [2]:
import os
import cv2
import torch
import wandb
import random
import torchvision
import numpy as np
from glob import glob
import seaborn as sns
from PIL import Image
import torch.nn as nn
from tqdm.auto import tqdm
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.nn.functional as F
# from google.colab import userdata
from datetime import datetime, date
from sklearn.model_selection import KFold
from torchvision import models, transforms
from torchvision import datasets, transforms, models
from torch.utils.data import (
    DataLoader, TensorDataset, Subset,
    ConcatDataset, Dataset
)
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

today = date.today()
curr_date = today.strftime("%d-%m-%Y")

In [3]:
# os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
# wandb.login()

In [4]:
# Ensure deterministic behavior

RANDOM_STATE = 42

torch.backends.cudnn.deterministic = True
random.seed(hash("setting random seeds") % 2**32 - 1)
np.random.seed(hash("improves reproducibility") % 2**32 - 1)
torch.manual_seed(hash("by removing stochasticity") % 2**32 - 1)
torch.cuda.manual_seed_all(hash("so runs are repeatable") % 2**32 - 1)

# Device configuration
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [5]:
CONFIG = dict({
    "project_name": f"EfficientNetB4_KFold_{curr_date}",
    "epochs": 1,
    "batch_size": 32,
    "lr": 0.0001,
    "k_folds": 5,
    "lr_scheduler_patience": 10,
    "lr_scheduler_factor": 0.1,
    "patience": 9,
    "num_classes": 3,
    "dataset_path": "/content/drive/MyDrive/Pest Classification/Dataset/castor_v2_224x224",
    "model_name": "Efficient_B4",
    "experimant_type": ["normal", "Img-Aug", "Cutmix", "Other"],
    "optimizer": "adam",  # ["adam", "adamw", "sgd"]
    "weight_decay": 0.001,
    "use_wandb": False   # False for Testing
})

In [6]:
# ---------------- DATASET ---------------- #
class PestDataset(Dataset):
    def __init__(self, files, labels, transform=None):
        self.files = files
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


# ---------------- LOAD FILES ---------------- #
def load_dataset(root):
    class_folders = sorted(os.listdir(root))
    files, labels = [], []

    for label, cls in enumerate(class_folders):
        img_files = glob(os.path.join(root, cls, "*"))
        files.extend(img_files)
        labels.extend([label] * len(img_files))

    return files, labels


# ---------------- MODEL ---------------- #
class EfficientNetB4Custom(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        base = models.efficientnet_b4(weights="IMAGENET1K_V1")
        for param in base.parameters():  # Freeze layers
            param.requires_grad = False

        self.features = base.features

        # EfficientNet-B0 output channels = 1280
        in_channels = 1792

        # 4. Custom classifier head (mirrors your TF layers)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),         # GlobalAveragePooling2D
            nn.Flatten(),

            nn.BatchNorm1d(in_channels),
            nn.Dropout(0.3),

            nn.Linear(in_channels, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),

            nn.Linear(512, 128),
            nn.ReLU(inplace=True),

            nn.Linear(128, num_classes)      # Softmax NOT added; CrossEntropy handles it
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

def get_model(num_classes=3):
    return EfficientNetB4Custom(num_classes)


# ---------------- CLASS WEIGHTS ---------------- #
def get_class_weights(labels):
    class_count = np.bincount(labels)
    weights = 1.0 / class_count
    weights = weights / weights.sum()
    return torch.tensor(weights, dtype=torch.float).to(device)

def get_optimer(optim, model, lr, weight_decay=0.01):
    if optim.lower() == "adam":
        return torch.optim.Adam(model.parameters(), lr, weight_decay=weight_decay)
    elif optim.lower() == "adamw":
        return torch.optim.AdamW(model.parameters(), lr, weight_decay=weight_decay)
    elif optim.lower() == "sgd":
        return torch.optim.SGD(model.parameters(), lr, momentum=0.9, weight_decay=weight_decay)
    else:
        raise ValueError("Not a valid Optimer Name!")

In [7]:
model = get_model(3)
total_params = sum(p.numel() for p in model.parameters())
print(total_params)

Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [00:00<00:00, 202MB/s]


18536267


In [8]:
def evaluate(model, loader, criterion):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    total_loss = 0

    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)

            outputs = model(imgs)
            loss = criterion(outputs, lbls)
            total_loss += loss.item()

            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())
            probs = torch.softmax(outputs, dim=1).detach().cpu().numpy()
            all_probs.extend(probs)

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    rec = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, output_dict=False, zero_division=0)

    return acc, prec, rec, f1, total_loss / len(loader), cm, report, all_labels, all_preds, all_probs

In [9]:
def plot_cm(cm, class_names, title):
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)

    img_path = f"{title.replace(' ', '_')}.png"
    plt.savefig(img_path, dpi=300, bbox_inches="tight")
    plt.close()
    return img_path


def train_fold(fold, train_idx, val_idx, class_names, group_name):

    if CONFIG["use_wandb"]:
        wandb.init(
            project=f"{CONFIG["project_name"]}",
            name=f"Fold-{fold+1}",
            group=group_name,
            config=CONFIG,
        )

    # Dataset split
    train_ds = Subset(dataset, train_idx)
    val_ds = Subset(dataset, val_idx)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False)

    model = get_model(CONFIG["num_classes"]).to(device)

    # Criterion
    class_weights = get_class_weights(np.array(labels)[train_idx])
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    # optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"])
    optimizer = get_optimer(CONFIG["optimizer"], model, CONFIG["lr"], CONFIG["weight_decay"])

    # Learning rate scheduler
    # scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    #     optimizer,
    #     mode='min',
    #     factor=CONFIG["lr_scheduler_factor"],
    #     patience=CONFIG["lr_scheduler_patience"]
    # )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer=optimizer,
        T_max=CONFIG["epochs"],
        eta_min=1e-6
    )

    best_acc = 0
    best_prec = 0
    best_rec = 0
    best_f1 = 0
    best_loss = float('inf')
    patience_counter = 0
    best_epoch = -1

    # For storing data of the best epoch
    best_val_labels, best_val_preds = None, None
    best_train_labels, best_train_preds = None, None
    best_val_cm, best_val_report = None, None # Initialize to None
    best_train_cm, best_train_report = None, None # Initialize to None

    for epoch in range(CONFIG["epochs"]):

        print(f"\n====== EPOCH {epoch+1}/{CONFIG['epochs']} ======")

        # ---------------- TRAIN ---------------- #
        model.train()
        train_loss_sum = 0
        all_train_preds, all_train_labels, all_train_probs = [], [], [] # Accumulate predictions and labels for training metrics

        progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']}", leave=False)

        for imgs, lbls in progress:
            imgs, lbls = imgs.to(device), lbls.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, lbls)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item()
            preds = outputs.argmax(1)
            all_train_preds.extend(preds.cpu().numpy())
            all_train_labels.extend(lbls.cpu().numpy())
            train_probs = torch.softmax(outputs, dim=1).detach().cpu().numpy()
            all_train_probs.extend(train_probs)

        train_loss = train_loss_sum / len(train_loader)
        train_acc = accuracy_score(all_train_labels, all_train_preds)
        train_prec = precision_score(all_train_labels, all_train_preds, average="macro", zero_division=0)
        train_rec = recall_score(all_train_labels, all_train_preds, average="macro", zero_division=0)
        train_f1 = f1_score(all_train_labels, all_train_preds, average="macro", zero_division=0)
        train_auc = roc_auc_score(all_train_labels, all_train_probs, multi_class="ovr")

        # ---------------- VALIDATION ---------------- #
        val_acc, val_prec, val_rec, val_f1, val_loss, \
        val_cm, val_report, v_labels, v_preds, v_probs = evaluate(model, val_loader, criterion)
        val_auc = roc_auc_score(v_labels, v_probs, multi_class="ovr")

        # Step the learning rate scheduler
        # scheduler.step(val_loss)
        scheduler.step()

        # WandB logs per epoch
        if CONFIG["use_wandb"]:
            wandb.log({
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "train_accuracy": train_acc,
                "train_precision": train_prec,
                "train_recall": train_rec,
                "train_auc": train_auc,
                "val_auc": val_auc,
                "train_f1": train_f1,
                "val_accuracy": val_acc,
                "val_precision": val_prec,
                "val_recall": val_rec,
                "val_f1": val_f1,
                "learning_rate": optimizer.param_groups[0]['lr']
            })

        print(f"Epoch {epoch+1}/{CONFIG['epochs']}")
        print(f"Train Accuracy: {train_acc:.4f} | Train Loss: {train_loss:.4f}")
        print(f"Test Accuracy: {val_acc:.4f} | Test Loss: {val_loss:.4f}")
        # ---------------- BEST EPOCH CHECK ---------------- #
        if val_acc > best_acc:
            best_acc = val_acc
            best_prec = val_prec
            best_rec = val_rec
            best_f1 = val_f1
            best_loss = val_loss
            best_roc = val_auc
            best_epoch = epoch + 1
            patience_counter = 0

            # Save model
            model_path = f"{CONFIG['model_name']}_best_fold{fold + 1}.pth"
            torch.save(model.state_dict(), model_path)
            if CONFIG["use_wandb"]:
                wandb.save(model_path)

            # Store validation info
            best_val_labels, best_val_preds = v_labels, v_preds
            best_val_cm, best_val_report = val_cm, val_report

            # Store training info for the best epoch
            best_train_labels, best_train_preds = all_train_labels, all_train_preds
            best_train_cm = confusion_matrix(best_train_labels, best_train_preds)
            best_train_report = classification_report(best_train_labels, best_train_preds, output_dict=False, zero_division=0)

        else:
            patience_counter += 1
            if patience_counter >= CONFIG["patience"]:
                print("EARLY STOPPING!")
                break

    # ---------------- LOGGING BEST EPOCH DATA ---------------- #

    print(f"\n===== BEST EPOCH = {best_epoch} | Best Val Acc = {best_acc:.4f} =====")

    # Generate heatmaps - Ensure best_val_cm and best_train_cm are not None
    if best_val_cm is not None and best_train_cm is not None:
        val_cm_img = plot_cm(best_val_cm, class_names, f"Fold{fold+1}_Validation_CM")
        train_cm_img = plot_cm(best_train_cm, class_names, f"Fold{fold+1}_Training_CM")

        # Log confusion matrices as images
        if CONFIG["use_wandb"]:
            wandb.log({
                "best_validation_confusion_matrix": wandb.Image(val_cm_img),
                "best_training_confusion_matrix": wandb.Image(train_cm_img),
                "best_validation_classification_report": wandb.Html(f"<pre>{best_val_report}</pre>"),
                "best_training_classification_report": wandb.Html(f"<pre>{best_train_report}</pre>")
            })
    else:
        print("Warning: No best model saved or metrics to log for best epoch.")

    if CONFIG["use_wandb"]:
        wandb.finish()

    return fold, best_epoch, best_acc, best_loss, best_prec, best_rec, best_f1, best_roc


In [10]:
if 'run' not in globals():
    run = 1

if __name__ == "__main__":
    # Preprocessing (ImageNet, no augmentation)
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    files, labels = load_dataset(CONFIG["dataset_path"])
    class_names = sorted(os.listdir(CONFIG["dataset_path"]))
    dataset = PestDataset(files, labels, transform)

    # Store best validation metrics of each fold
    all_fold_val_acc = []
    all_fold_val_loss = []
    all_fold_val_prec = []
    all_fold_val_rec = []
    all_fold_val_f1 = []
    all_fold_val_roc = []

    kf = KFold(n_splits=CONFIG["k_folds"], shuffle=True, random_state=RANDOM_STATE)

    for fold, (train_idx, val_idx) in enumerate(kf.split(files)):
        print(f"\n\n========== FOLD {fold+1} ==========")
        group_name = f"{CONFIG['experimant_type'][0]}_{CONFIG["optimizer"]}_momentum+lr"

        if CONFIG["use_wandb"]:
            fold_results_table = wandb.Table(
                columns=[
                    "fold",
                    "best_epoch",
                    "best_val_accuracy",
                    "best_val_loss",
                    "best_precision",
                    "best_recall",
                    "best_f1",
                    "best_roc_auc"
            ])

        fold_num, best_epoch, best_acc, best_loss, best_prec, best_rec, best_f1, best_roc = train_fold(fold, train_idx, val_idx, class_names, group_name)

        if CONFIG["use_wandb"]:
            fold_results_table.add_data(
                fold_num + 1,
                best_epoch,
                best_acc,
                best_loss,
                best_prec,
                best_rec,
                best_f1,
                best_roc
            )
        # wandb.log({f"Fold_{fold+1}_Summary": fold_results_table})

        all_fold_val_acc.append(best_acc)
        all_fold_val_loss.append(best_loss)
        all_fold_val_prec.append(best_prec)
        all_fold_val_rec.append(best_rec)
        all_fold_val_f1.append(best_f1)
        all_fold_val_roc.append(best_roc)


    run+=1

    print("\n====== FINAL K-FOLD RESULTS (AVG OVER ALL FOLDS) ======")
    print(f"Average Validation Accuracy : {np.mean(all_fold_val_acc):.4f}")
    print(f"Average Validation Loss     : {np.mean(all_fold_val_loss):.4f}")
    print(f"Average Validation Precision: {np.mean(all_fold_val_prec):.4f}")
    print(f"Average Validation Recall   : {np.mean(all_fold_val_rec):.4f}")
    print(f"Average Validation F1 Score : {np.mean(all_fold_val_f1):.4f}")
    print(f"Average Validation ROC AUC  : {np.mean(all_fold_val_roc):.4f}")





========== FOLD 1 ==========

====== EPOCH 1/1 ======


Epoch 1/1:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch 1/1
Train Accuracy: 0.6881 | Train Loss: 1.0049
Test Accuracy: 0.8241 | Test Loss: 0.9950

===== BEST EPOCH = 1 | Best Val Acc = 0.8241 =====


========== FOLD 2 ==========

====== EPOCH 1/1 ======


Epoch 1/1:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch 1/1
Train Accuracy: 0.6557 | Train Loss: 1.0064
Test Accuracy: 0.8232 | Test Loss: 0.9767

===== BEST EPOCH = 1 | Best Val Acc = 0.8232 =====


========== FOLD 3 ==========

====== EPOCH 1/1 ======


...

Epoch 1/1
Train Accuracy: 0.6671 | Train Loss: 0.9851
Test Accuracy: 0.8131 | Test Loss: 0.9565

===== BEST EPOCH = 1 | Best Val Acc = 0.8131 =====


========== FOLD 4 ==========

====== EPOCH 1/1 ======


Epoch 1/1:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch 1/1
Train Accuracy: 0.5158 | Train Loss: 1.0151
Test Accuracy: 0.8131 | Test Loss: 1.0012

===== BEST EPOCH = 1 | Best Val Acc = 0.8131 =====


========== FOLD 5 ==========

====== EPOCH 1/1 ======


Epoch 1/1:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch 1/1
Train Accuracy: 0.6255 | Train Loss: 1.0091
Test Accuracy: 0.8384 | Test Loss: 0.9982

===== BEST EPOCH = 1 | Best Val Acc = 0.8384 =====

====== FINAL K-FOLD RESULTS (AVG OVER ALL FOLDS) ======
Average Validation Accuracy : 0.8224
Average Validation Loss     : 0.9855
Average Validation Precision: 0.8156
Average Validation Recall   : 0.7651
Average Validation F1 Score : 0.7554
Average Validation ROC AUC  : 0.9447


In [ ]:
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 kB 16.9 MB/s eta 0:00:00


In [ ]:
model = get_model(3).to(device)
path = "/content/Efficient_B4_best_fold3.pth"
model.load_state_dict(torch.load(path, map_location=torch.device('cpu')))
model.eval()

dummy_input = torch.randn(1, 3, 224, 224).to(device)

torch.onnx.export(
    model,
    dummy_input,
    "EfficientNetB4.onnx",
    opset_version=18,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch"},
        "output": {0: "batch"}
    },
    external_data=False
)